### Context management
Context is an overloaded term. There are two main classes of context you might care about:

1. Context available locally to your code: this is data and dependencies you might need when tool functions run, during callbacks like on_handoff, in lifecycle hooks, etc.
2. Context available to LLMs: this is data the LLM sees when generating a response.

[See the details](https://openai.github.io/openai-agents-python/context/#use-local-context-for-capability-visibility)

In [2]:
from agents import (Agent, 
                    Runner, AsyncOpenAI, 
                    OpenAIChatCompletionsModel, set_tracing_disabled,
                    RunContextWrapper)

from agents.decorators import tool

In [3]:
# Tracing disabled
set_tracing_disabled(disabled=True)

In [4]:
import os
gemini_api = os.environ['GEMINI_API_KEY']

In [5]:
# 1. Which LLM Service?
external_client = AsyncOpenAI(
    api_key=gemini_api,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [6]:
# 2. Which LLM Model?
llm_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=external_client
)

##### Step-1

In [40]:
# from dataclasses import dataclass
from pydantic import BaseModel

class UserInfo(BaseModel):  
    name: str
    uid: int

##### Step-2

In [46]:
@tool
def fetch_user_age(wrapper: RunContextWrapper[UserInfo]) -> str:  
    """Fetch the age of the user. Call this function to get user's age information."""
    return f"The user {wrapper.context.name} is 47 years old"

##### Step-3

In [47]:
user_info = UserInfo(name="John", uid=123)

##### Step-4

In [48]:
agent = Agent[UserInfo](  
    name="Assistant",
    tools=[fetch_user_age],
    tool_use_behavior="stop_on_first_tool"
)

##### Step-5

In [49]:
result = await Runner.run(  
    starting_agent=agent,
    input="What is the age of the user?",
    context=user_info,
)

In [50]:
print(result.final_output)

The user John is 47 years old
